# Ensemble Methods Practical (90 minutes)

**Learning Objectives:**
- Build and compare bagging and boosting ensembles on a real dataset
- Practice bias-variance diagnostics using cross-validation and hold-out metrics
- Tune ensemble hyperparameters for higher recall on the minority class
- Combine diverse learners via stacking and interpret ensemble behavior

**What You'll Do:**
1. Explore and preprocess the diamonds classification dataset
2. Train a baseline Random Forest and evaluate performance
3. Tune hyperparameters and benchmark gradient boosting
4. Build a stacking ensemble and compare against individual models
5. Summarize findings and recommend a production-ready model

**Structure:**
- ✅ Pre-filled utilities and scaffolding
- ✏️ TODO cells where you implement key steps

> **Tip:** Work sequentially. Later sections depend on earlier outputs.

## 1. Setup & Imports

In [ ]:
# Install necessary libraries
!pip install -q xgboost catboost lightgbm umap-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, mean_squared_error, r2_score, classification_report
)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor

# Set random state for reproducibility
RANDOM_STATE = 42

## Part 1: Classification - Telco Customer Churn

We will use the Telco Customer Churn dataset. The goal is to predict whether a customer will churn (leave the service).

### 1.1 Data Loading & Preprocessing (Pre-filled)

In [ ]:
# Load dataset from a public URL
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Basic cleaning
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])
df = df.drop(columns=['customerID'])

# Encode target
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Identify columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
numeric_cols.remove('Churn')

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric columns: {numeric_cols}")

# Preprocessing Pipeline
# 1. Encode categorical variables
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# 2. Split data
X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# 3. Scale numeric features
scaler = StandardScaler()
# We need to identify the indices or names of numeric columns in the encoded dataframe
# Since get_dummies puts encoded columns at the end or replaces them, let's find numeric cols again in X
numeric_features_in_X = [col for col in numeric_cols if col in X.columns]

X_train[numeric_features_in_X] = scaler.fit_transform(X_train[numeric_features_in_X])
X_test[numeric_features_in_X] = scaler.transform(X_test[numeric_features_in_X])

print("Data shapes:")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

### 1.2 Exploratory Data Analysis with UMAP (Pre-filled)

Uniform Manifold Approximation and Projection (UMAP) is a dimension reduction technique that can be used for visualization similar to t-SNE, but often faster and better at preserving global structure.

In [ ]:
# UMAP Visualization
print("Computing UMAP embedding...")
reducer = umap.UMAP(random_state=RANDOM_STATE)
embedding = reducer.fit_transform(X_train)

plt.figure(figsize=(10, 8))
plt.scatter(
    embedding[:, 0],
    embedding[:, 1],
    c=y_train,
    cmap='Spectral',
    s=5,
    alpha=0.6
)
plt.colorbar(label='Churn')
plt.title('UMAP projection of the Telco Churn dataset')
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.show()

### 1.3 Model Training & Comparison

In this section, you will train four different ensemble models:
1.  **Random Forest**
2.  **XGBoost**
3.  **CatBoost**
4.  **LightGBM**

For each model, you should:
- Initialize the classifier.
- Train it on the training data.
- Predict probabilities on the test data.

#### Helper Function: Density Plot
Use this function to visualize the separation of classes by predicted probabilities.

In [ ]:
def plot_density(y_test, y_probs, model_name="Model"):
    plt.figure(figsize=(8, 6))
    sns.kdeplot(y_probs[y_test == 0], label='Class 0 (No Churn)', fill=True, color='blue', alpha=0.3)
    sns.kdeplot(y_probs[y_test == 1], label='Class 1 (Churn)', fill=True, color='red', alpha=0.3)
    plt.xlabel('Predicted Probability')
    plt.ylabel('Density')
    plt.title(f'Density Plot for Predicted Probabilities - {model_name}')
    plt.legend()
    plt.show()

#### 1.3.1 Random Forest

In [ ]:
# TODO: Initialize and train a Random Forest Classifier
# 1. Instantiate RandomForestClassifier with 100 estimators and the random state.
# 2. Fit the model on the training data (X_train, y_train).
# 3. Predict class probabilities on the test set (X_test) using .predict_proba().
#    Store the probabilities for the positive class (column 1) in a variable named 'rf_probs'.

rf_model = None # Initialize your model here
rf_probs = None # Store probabilities here

#### 1.3.2 XGBoost

In [ ]:
# TODO: Initialize and train XGBoost Classifier
# 1. Instantiate XGBClassifier.
#    - Set use_label_encoder=False to avoid warnings.
#    - Set eval_metric='logloss'.
#    - Use the defined RANDOM_STATE.
# 2. Fit the model on the training data.
# 3. Predict class probabilities on the test set.

xgb_model = None 
xgb_probs = None

#### 1.3.3 CatBoost

In [ ]:
# TODO: Initialize and train CatBoost Classifier
# 1. Instantiate CatBoostClassifier.
#    - Set verbose=0 to suppress training output.
#    - Use the defined RANDOM_STATE.
# 2. Fit the model on the training data.
# 3. Predict class probabilities on the test set.

cb_model = None
cb_probs = None

#### 1.3.4 LightGBM

In [ ]:
# TODO: Initialize and train LightGBM Classifier
# 1. Instantiate LGBMClassifier with the defined RANDOM_STATE.
# 2. Fit the model on the training data.
# 3. Predict class probabilities on the test set.

lgbm_model = None
lgbm_probs = None

### 1.4 Evaluation and Comparison

Compare the models using ROC-AUC score and visualize the density plots.

In [ ]:
# TODO: Calculate and print ROC-AUC scores
# 1. Create a dictionary or list to store model names and their corresponding ROC-AUC scores.
#    - Use roc_auc_score(y_true, y_score) for each model.
#    - Remember to use the probabilities for the positive class (column 1).
# 2. Print the scores for comparison.

In [ ]:
# TODO: Visualize predicted probabilities
# Use the `plot_density` helper function defined earlier to visualize the distribution 
# of predicted probabilities for your best performing model (or all of them).
# Example: plot_density(y_test, rf_probs[:, 1], "Random Forest")

### 1.5 Cross-Validation Comparison

Evaluate the stability of the models using Cross-Validation. This gives a better estimate of model performance on unseen data.

In [ ]:
# TODO: Compare models using Cross-Validation
# 1. Create a list of models to evaluate.
# 2. Iterate through the models and perform 5-fold Cross-Validation using `cross_val_score`.
#    - Use 'roc_auc' as the scoring metric.
# 3. Store the results (scores) and model names.
# 4. Print the mean and standard deviation of the scores for each model.
# 5. Create a boxplot to visually compare the distribution of CV scores across models.

### 1.6 Detailed Performance Analysis

Go beyond ROC-AUC. Analyze the model's performance using:
1.  **Classification Report**: Precision, Recall, F1-Score for each class.
2.  **Confusion Matrix**: To see where the model is making errors (False Positives vs False Negatives).
3.  **Precision-Recall Curve**: Especially useful for imbalanced datasets like Churn.

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, PrecisionRecallDisplay

# TODO: Detailed Performance Analysis
# 1. Select your best performing model.
# 2. Convert predicted probabilities to binary predictions (0 or 1) using a threshold (e.g., 0.5).
# 3. Print the `classification_report` to see Precision, Recall, and F1-Score.
# 4. Compute and plot the `confusion_matrix` using `ConfusionMatrixDisplay`.
# 5. Plot the Precision-Recall Curve using `PrecisionRecallDisplay`.

## Part 2: Regression - California Housing

We will use the California Housing dataset to predict median house values.

In [ ]:
from sklearn.datasets import fetch_california_housing

# Load data
housing = fetch_california_housing()
X_reg = pd.DataFrame(housing.data, columns=housing.feature_names)
y_reg = housing.target

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE)

# Scale data
scaler_reg = StandardScaler()
X_train_reg = scaler_reg.fit_transform(X_train_reg)
X_test_reg = scaler_reg.transform(X_test_reg)

print(f"Regression Train shape: {X_train_reg.shape}")
print(f"Regression Test shape: {X_test_reg.shape}")

### 2.1 Train Ensemble Regressors
Choose one ensemble method (e.g., XGBoost or Random Forest) and train a regressor.

In [ ]:
# TODO: Train an Ensemble Regressor (e.g., XGBRegressor)
# 1. Instantiate the regressor with the defined RANDOM_STATE.
# 2. Fit the model on the regression training data (X_train_reg, y_train_reg).
# 3. Predict target values on the test set (X_test_reg).

reg_model = None 
y_pred_reg = None

# TODO: Evaluate the model
# 1. Calculate the Root Mean Squared Error (RMSE).
# 2. Calculate the R2 Score.
# 3. Print both metrics.

### 2.2 Feature Importance (Regression)
Visualize which features were most important for the regression model.

In [ ]:
# TODO: Visualize Feature Importance
# 1. Access the `feature_importances_` attribute of your trained regressor.
# 2. Sort the features by importance.
# 3. Create a horizontal bar plot (`plt.barh`) to display the importance of each feature.

In [ ]:
# TODO: Residual Analysis
# 1. Calculate residuals (Actual values - Predicted values).
# 2. Create a figure with two subplots.
#    - Subplot 1: Scatter plot of Predicted Values vs. Residuals. Add a horizontal line at y=0.
#      (This checks for homoscedasticity).
#    - Subplot 2: Histogram (or KDE plot) of the residuals.
#      (This checks if residuals are normally distributed).
# 3. Display the plots.

### 2.3 Residual Analysis

Analyze the residuals to check for patterns. Ideally, residuals should be randomly distributed around zero.
1. **Residuals vs Predicted Plot**: Checks for homoscedasticity (constant variance).
2. **Residual Distribution**: Checks if residuals are normally distributed.